In [1]:
# =============================================
# SECTION 1 — 3.3 Part 2: PCA with MLE Covariance (COMMON USERS ONLY):
#  1) Load ratings.csv + target_items.csv (I1/I2) from Section 3.1
#  2) Build users x {I1,I2} rating matrix (NaNs kept)
#  3) Compute MLE covariance matrix using ONLY users who rated both items (per pair)
#  4) Determine top-5 and top-10 peers (for 2 items, peers are identical)
#  5) Predict missing ratings using covariance-weighted centered peer ratings
#  6) Save:
#       - covariance_matrix_mle.csv
#       - predictions_mle_top5.csv
#       - predictions_mle_top10.csv
#       - compare_mle_top5_vs_top10.csv
#       - compare_part1_vs_part2_top10.csv
# =============================================

import os
import numpy as np
import pandas as pd

# --- Path ---
from pathlib import Path
PROJECT_ROOT = Path.cwd()

def resolve_existing(*candidates: str) -> str:
    for c in candidates:
        p = (PROJECT_ROOT / c)
        if p.exists():
            return str(p)
    tried = "\n".join([f"- {PROJECT_ROOT / c}" for c in candidates])
    raise FileNotFoundError(f"Could not find required file. Tried:\n{tried}")

# Dataset + section outputs
RATINGS_PATH = resolve_existing(
    r"C:\ml-20m\ml-20m\ratings.csv",)

OUT_DIR = resolve_existing(
    "SECTION1_DimensionalityReduction/data",
    "output",
    "outputs",
    ".",
)

# Files from Part 1 (for comparison in Point 9)
PART1_TOP10_PATH = resolve_existing(
    "SECTION1_DimensionalityReduction/data/predictions_top10.csv",
    "predictions_top10.csv",
)

# Target items from Section 3.1
TARGET_ITEMS_PATH = resolve_existing(
    "SECTION1_DimensionalityReduction/data/target_items.csv",
    "target_items.csv",
)

os.makedirs(OUT_DIR, exist_ok=True)
print("Ratings:", RATINGS_PATH)
print("Target items:", TARGET_ITEMS_PATH)
print("Output dir:", OUT_DIR)

# =============================================
# 1) Load data + get I1/I2 movieIds
# =============================================
ratings = pd.read_csv(RATINGS_PATH, usecols=["userId", "movieId", "rating"])
targets_items = pd.read_csv(TARGET_ITEMS_PATH)

I1 = int(targets_items.loc[targets_items["item_label"] == "I1_low_popularity", "movieId"].values[0])
I2 = int(targets_items.loc[targets_items["item_label"] == "I2_high_popularity", "movieId"].values[0])
target_items = [I1, I2]

print("I1 movieId =", I1)
print("I2 movieId =", I2)

# =============================================
# 2) Build user-item matrix (NaNs kept)
# =============================================
rm = (
    ratings[ratings["movieId"].isin(target_items)]
      .pivot_table(index="userId", columns="movieId", values="rating", aggfunc="mean")
      .sort_index()
)

# Rename columns to friendly labels (I1/I2) but keep mapping for saving
col_map = {I1: "I1", I2: "I2"}
rm = rm.rename(columns=col_map)

print("Rating matrix shape (users x 2 items):", rm.shape)
display(rm.head())

# =============================================
# 3) Item means (from observed ratings only) + centered diff matrix (NaNs kept)
# =============================================
item_means = rm.mean(axis=0, skipna=True)  # mean per item over observed only
diff = rm.copy()
for c in diff.columns:
    diff[c] = diff[c] - item_means[c]  # NaNs remain NaNs

means_df = item_means.reset_index()
means_df.columns = ["item_label", "mean_rating"]
means_df.to_csv(os.path.join(OUT_DIR, "target_item_means_part2.csv"), index=False)

# =============================================
# 4) MLE covariance (common users only)
#    cov(i,j) = mean( (ri-mean_i)*(rj-mean_j) ) over users with both ratings
# =============================================
items = list(diff.columns)  # ["I1","I2"]
cov_mle = pd.DataFrame(np.zeros((len(items), len(items))), index=items, columns=items, dtype=float)

for i in items:
    for j in items:
        # users who rated BOTH i and j
        common = diff[[i, j]].dropna()
        if len(common) == 0:
            cov = 0.0
        else:
            cov = float(np.mean(common[i].values * common[j].values))
        cov_mle.loc[i, j] = cov

cov_path = os.path.join(OUT_DIR, "covariance_matrix_mle.csv")
cov_mle.to_csv(cov_path)
print("Saved:", cov_path)
display(cov_mle)

# =============================================
# 5) Top-N peers (by |covariance|), excluding self
# =============================================
def top_peers(item_label: str, N: int) -> list[str]:
    s = cov_mle.loc[item_label].drop(index=item_label)
    s = s.reindex(s.abs().sort_values(ascending=False).index)
    return s.head(N).index.tolist()

peers_5  = {it: top_peers(it, 5) for it in items}
peers_10 = {it: top_peers(it, 10) for it in items}
print("Top-5 peers:", peers_5)
print("Top-10 peers:", peers_10)

# NOTE: With only 2 items, peers_5 and peers_10 will be identical (each item has only 1 peer).

# =============================================
# 6) Predict missing ratings using covariance-weighted centered peer ratings
#    pred(u,i) = mean_i + sum_j cov(i,j) * diff(u,j) / sum_j |cov(i,j)|
#    If denom==0 OR user has no peer ratings => fallback to mean_i
# =============================================
def predict_missing(peers_dict: dict[str, list[str]]) -> pd.DataFrame:
    rows = []
    for user_id, row in rm.iterrows():
        for it in items:
            if pd.isna(row[it]):  # only predict missing entries
                peers = peers_dict[it]
                num = 0.0
                denom = 0.0
                used = 0
                for pj in peers:
                    v = diff.loc[user_id, pj]
                    if pd.notna(v):
                        w = float(cov_mle.loc[it, pj])
                        num += w * float(v)
                        denom += abs(w)
                        used += 1
                if denom == 0 or used == 0:
                    pred = float(item_means[it])
                    reason = "fallback_mean"
                else:
                    pred = float(item_means[it] + (num / denom))
                    reason = "cov_weighted"
                rows.append({
                    "userId": int(user_id),
                    "item_label": it,
                    "itemId": int(I1 if it == "I1" else I2),
                    "predicted_rating": round(pred, 4),
                    "reason": reason
                })
    return pd.DataFrame(rows)

pred5 = predict_missing(peers_5)
pred10 = predict_missing(peers_10)

p_mle5  = os.path.join(OUT_DIR, "predictions_mle_top5.csv")
p_mle10 = os.path.join(OUT_DIR, "predictions_mle_top10.csv")
pred5.to_csv(p_mle5, index=False)
pred10.to_csv(p_mle10, index=False)
print("Saved:", p_mle5)
print("Saved:", p_mle10)

display(pred5.head())


Ratings: C:\ml-20m\ml-20m\ratings.csv
Target items: c:\Users\Ahmed elmasry\OneDrive - GALALA University\Desktop\Section1\SECTION1_DimensionalityReduction\data\target_items.csv
Output dir: c:\Users\Ahmed elmasry\OneDrive - GALALA University\Desktop\Section1\SECTION1_DimensionalityReduction\data
I1 movieId = 118758
I2 movieId = 235
Rating matrix shape (users x 2 items): (16419, 2)


movieId,I2,I1
userId,,
5,3.0,NaN
20,3.5,NaN
21,3.0,NaN
23,3.0,NaN
27,3.0,NaN


Saved: c:\Users\Ahmed elmasry\OneDrive - GALALA University\Desktop\Section1\SECTION1_DimensionalityReduction\data\covariance_matrix_mle.csv


,I2,I1
I2,0.917246,0.0
I1,0.000000,0.0


Top-5 peers: {'I2': ['I1'], 'I1': ['I2']}
Top-10 peers: {'I2': ['I1'], 'I1': ['I2']}
Saved: c:\Users\Ahmed elmasry\OneDrive - GALALA University\Desktop\Section1\SECTION1_DimensionalityReduction\data\predictions_mle_top5.csv
Saved: c:\Users\Ahmed elmasry\OneDrive - GALALA University\Desktop\Section1\SECTION1_DimensionalityReduction\data\predictions_mle_top10.csv


,userId,item_label,itemId,predicted_rating,reason
0,5,I1,118758,1.5,fallback_mean
1,20,I1,118758,1.5,fallback_mean
2,21,I1,118758,1.5,fallback_mean
3,23,I1,118758,1.5,fallback_mean
4,27,I1,118758,1.5,fallback_mean


In [2]:
# =============================================
# 7) Comparisons required by the spec
#   - Compare MLE top-5 vs top-10 
#   - Compare Part 1 (mean-fill) vs Part 2 (MLE), both top-10  [Point 9]
# =============================================

import os
import pandas as pd
import numpy as np

# Load saved predictions from this notebook
df_mle5  = pd.read_csv(os.path.join(OUT_DIR, "predictions_mle_top5.csv"))
df_mle10 = pd.read_csv(os.path.join(OUT_DIR, "predictions_mle_top10.csv"))

# ---- Compare MLE Top-5 vs Top-10 ----
m5 = df_mle5.rename(columns={"predicted_rating": "pred_mle_top5"})
m10 = df_mle10.rename(columns={"predicted_rating": "pred_mle_top10"})

cmp_mle = pd.merge(
    m5[["userId", "itemId", "item_label", "pred_mle_top5"]],
    m10[["userId", "itemId", "item_label", "pred_mle_top10"]],
    on=["userId", "itemId", "item_label"],
    how="inner"
)

cmp_mle["abs_diff"] = (cmp_mle["pred_mle_top5"] - cmp_mle["pred_mle_top10"]).abs()
cmp_mle_path = os.path.join(OUT_DIR, "compare_mle_top5_vs_top10.csv")
cmp_mle.to_csv(cmp_mle_path, index=False)

print("Saved:", cmp_mle_path)
print("MLE top5 vs top10 -> mean abs diff:", float(cmp_mle["abs_diff"].mean()))
print("MLE top5 vs top10 -> max abs diff :", float(cmp_mle["abs_diff"].max()))
display(cmp_mle.head())

# ---- Compare Part 1 vs Part 2 (both Top-10) ----
# Part 1 file is expected from your Part 1 notebook:
#   SECTION1_DimensionalityReduction/data/predictions_top10.csv
part1_top10 = pd.read_csv(PART1_TOP10_PATH)

# Normalize Part 1 columns (it uses itemId for movieId and predicted_rating column)
if "predicted_rating" in part1_top10.columns:
    part1 = part1_top10.copy()
elif "pred_top10" in part1_top10.columns:
    part1 = part1_top10.rename(columns={"pred_top10": "predicted_rating"})
else:
    part1 = part1_top10.copy()

part1 = part1.rename(columns={"predicted_rating": "pred_part1_top10"})
part2 = df_mle10.rename(columns={"predicted_rating": "pred_part2_mle_top10"})

cmp_p1_p2 = pd.merge(
    part1[["userId", "itemId", "pred_part1_top10"]],
    part2[["userId", "itemId", "pred_part2_mle_top10"]],
    on=["userId", "itemId"],
    how="inner"
)

cmp_p1_p2["abs_diff"] = (cmp_p1_p2["pred_part1_top10"] - cmp_p1_p2["pred_part2_mle_top10"]).abs()

cmp_p1_p2_path = os.path.join(OUT_DIR, "compare_part1_vs_part2_top10.csv")
cmp_p1_p2.to_csv(cmp_p1_p2_path, index=False)

print("Saved:", cmp_p1_p2_path)
print("Part1 vs Part2 (top10) -> mean abs diff:", float(cmp_p1_p2["abs_diff"].mean()))
print("Part1 vs Part2 (top10) -> max abs diff :", float(cmp_p1_p2["abs_diff"].max()))
display(cmp_p1_p2.head())


Saved: c:\Users\Ahmed elmasry\OneDrive - GALALA University\Desktop\Section1\SECTION1_DimensionalityReduction\data\compare_mle_top5_vs_top10.csv
MLE top5 vs top10 -> mean abs diff: 0.0
MLE top5 vs top10 -> max abs diff : 0.0


,userId,itemId,item_label,pred_mle_top5,pred_mle_top10,abs_diff
0,5,118758,I1,1.5,1.5,0.0
1,20,118758,I1,1.5,1.5,0.0
2,21,118758,I1,1.5,1.5,0.0
3,23,118758,I1,1.5,1.5,0.0
4,27,118758,I1,1.5,1.5,0.0


Saved: c:\Users\Ahmed elmasry\OneDrive - GALALA University\Desktop\Section1\SECTION1_DimensionalityReduction\data\compare_part1_vs_part2_top10.csv
Part1 vs Part2 (top10) -> mean abs diff: 0.0
Part1 vs Part2 (top10) -> max abs diff : 0.0


,userId,itemId,pred_part1_top10,pred_part2_mle_top10,abs_diff
0,5,118758,1.5,1.5,0.0
1,20,118758,1.5,1.5,0.0
2,21,118758,1.5,1.5,0.0
3,23,118758,1.5,1.5,0.0
4,27,118758,1.5,1.5,0.0
